# Structured / Constrained Generation — Hands-On

**LLM Engineering · Domain 2 · Roadmap Weeks 11/15**

Companion to `02 Literature Notes/LLM Engineering/Structured Generation`. Runs offline with a fake model.

## 0. A fake model that sometimes emits bad JSON

In [ ]:
%pip install -q numpy
import json, random
random.seed(0)
def fake_model(prompt):
    # 40% chance of malformed output unless the prompt mentions 'Fix'
    if "Invalid" in prompt:
        return '{"vendor": "Acme", "total": 120.0, "line_items": ["widget"]}'
    return random.choice([
        '{"vendor": "Acme", "total": 120.0, "line_items": ["widget"]}',
        '```json\n{"vendor": "Acme", "total": -5, "line_items": []}\n```',
        'Sure! Here is your invoice: {vendor: Acme}',   # not JSON
    ])
print("ok")

## 1. Validate: structural + semantic

In [ ]:
def strip_fences(s):
    s = s.strip()
    if s.startswith("```"):
        s = s.split("```")[1]
        s = s[4:] if s.startswith("json") else s
    return s.strip()

def validate(raw):
    data = json.loads(strip_fences(raw))          # structural
    assert isinstance(data.get("vendor"), str)
    assert isinstance(data.get("total"), (int, float))
    assert isinstance(data.get("line_items"), list)
    assert data["total"] >= 0, "total must be non-negative"   # semantic
    return data

print(validate('{"vendor":"A","total":10,"line_items":[]}'))

## 2. Validate-and-repair loop with bounded retries

In [ ]:
def generate_structured(call_model, prompt, max_retries=4):
    err = None
    for attempt in range(max_retries):
        msg = prompt if err is None else f"{prompt}\nInvalid: {err}\nReturn ONLY corrected JSON."
        raw = call_model(msg)
        try:
            obj = validate(raw)
            print(f"  attempt {attempt+1}: OK")
            return obj
        except Exception as e:
            print(f"  attempt {attempt+1}: {type(e).__name__}: {e}")
            err = str(e)
    raise ValueError(f"failed after {max_retries}: {err}")

print("result:", generate_structured(fake_model, "Extract the invoice as JSON."))

## 3. Constrained decoding guarantees structure by construction

In [ ]:
import numpy as np
VOCAB = ['{', '"ok"', ':', 'true', 'false', '}']
STEP_LEGAL = {0:[0],1:[1],2:[2],3:[3,4],4:[5]}
def constrained_decode(value_pref):
    out=[]
    for step in range(5):
        logits = np.zeros(len(VOCAB)); 
        if step==3: logits[value_pref]=5   # model 'wants' true or false
        legal = STEP_LEGAL[step]
        masked = np.full_like(logits, -np.inf); masked[legal]=logits[legal]
        out.append(int(masked.argmax()))
    return "".join(VOCAB[t] for t in out)
print("prefers true :", constrained_decode(3))
print("prefers false:", constrained_decode(4))
print("both always valid JSON -> invalid output is unreachable")

## 4. Structural validity is not semantic correctness

In [ ]:
well_formed = '{"vendor":"Acme","total":999,"line_items":["a"]}'
data = json.loads(well_formed)
# total (999) != sum of line items -> parses fine but is WRONG
line_sum = 10  # pretend the one item costs 10
print("parses:", True, " semantically consistent:", data['total']==line_sum)

## 5. Exercises
1. Add a cross-field check: total == sum(prices). Trigger a repair on mismatch.
2. Extend the grammar to numbers (digits + optional decimal).
3. Swap the dataclass for a real Pydantic model and use `model_validate_json`.
4. Add a retry budget/cost counter and log attempts.

## Links
- Literature note: `02 Literature Notes/LLM Engineering/Structured Generation`
- Snippets: `04 Code Snippets/LLM/Validate and Repair Loop for JSON Output`, `.../Constrained Decoding with a Token Mask`
- MOC: `06 Maps of Content/LLM Engineering Concepts`